In [ ]:
import os

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from utils.data import MalwareDataset

data = np.array(MalwareDataset())

In [ ]:
MAX_LENGTH = 64


class ByteDataset(Dataset):
    def __init__(self, paths, labels):
        assert len(paths) == len(
            labels
        ), "Number of file paths should match number of labels"
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx: int):
        if idx > len(self):
            raise ValueError("Given index exceeds total number of samples")

        path = self.paths[idx]
        label = self.labels[idx]

        n_bytes = os.path.getsize(path)
        offset = np.random.randint(0, n_bytes - MAX_LENGTH)

        with open(path, "rb") as file:
            file.seek(offset)
            byte_sequence = file.read(MAX_LENGTH)

        return np.frombuffer(byte_sequence, dtype=np.uint8), label


byte_ds = ByteDataset(data[:, 0], data[:, 1])
byte_ds[0]

(array([101,  32, 116, 111,  32, 105, 110, 105, 116, 105,  97, 108, 105,
        122, 101,  32, 104, 101,  97, 112,  13,  10,   0,   0,   0,   0,
         82,  54,  48,  50,  55,  13,  10,  45,  32, 110, 111, 116,  32,
        101, 110, 111, 117, 103, 104,  32, 115, 112,  97,  99, 101,  32,
        102, 111, 114,  32, 108, 111, 119, 105, 111,  32, 105, 110],
       dtype=uint8),
 np.str_('1'))

In [ ]:
class MaskedByteDataset(Dataset):
    def __init__(self, paths, labels, mask_prob: float = 0.15):
        self.byte_ds = ByteDataset(paths, labels)
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.byte_ds)

    def __getitem__(self, idx):
        byte_seq, label = self.byte_ds[idx]

        tokens = np.full_like(byte_seq, -1)

        for i, token in enumerate(byte_seq):
            p = np.random.random()

            if p > self.mask_prob:
                tokens[i] = token
                q = np.random.random()

                if q < 0.8:
                    # Replace token with mask (80%)
                    processed[i] = tokenizer.encode("[MASK]")[0]
                elif 0.8 < q < 0.9:
                    # Replace token with random token (10%)
                    processed[i] = np.random.randint(2, self.vocab_size)
                else:
                    # Leave as is
                    pass